# JetBot Camera Reset

Fixes the two most common causes of JetBot camera failures, run from **inside** the Jupyter container:

1. A stale/orphaned Jupyter kernel still holding the Argus camera lock (`RuntimeError: Could not read image from camera.`)
2. A stale GStreamer plugin registry cache (`no element 'nvarguscamerasrc'` even though the plugin is installed)

Run the cells top to bottom. This notebook's own kernel is **not** killed.

In [ ]:
import os, subprocess, shutil, signal

def sh(cmd, timeout=60):
    """Run a command (string or list), return (returncode, combined output). Python 3.6 compatible."""
    p = subprocess.run(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, universal_newlines=True, timeout=timeout)
    return p.returncode, p.stdout


In [ ]:
# Step 1: Kill stale Jupyter kernels (releases the Argus camera lock).
# Skips THIS notebook's kernel so the cell doesn't kill itself.
my_pid = os.getpid()
rc, out = sh(["pgrep", "-f", "ipykernel_launcher"])   # list form: no shell wrapper to self-match
pids = [int(p) for p in out.split() if p.isdigit() and int(p) != my_pid]

if pids:
    for pid in pids:
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    print(f"Killed {len(pids)} stale kernel(s): {pids}")
else:
    print("No other running kernels found.")


In [ ]:
# Step 2: Clear the GStreamer plugin registry cache
cache = "/root/.cache/gstreamer-1.0"
shutil.rmtree(cache, ignore_errors=True)
print("Cache cleared." if not os.path.exists(cache) else "Failed to clear cache -- check permissions.")


In [ ]:
# Step 3: Verify the camera pipeline
rc, out = sh("gst-launch-1.0 nvarguscamerasrc num-buffers=1 ! fakesink", timeout=90)

if rc == 0:
    print("Camera check: OK -- device is free and responding.")
elif "Connection refused" in out:
    print("Detected stale Argus socket (nvargus-daemon restarted after the container started).")
    print("This can't be fixed from inside the container. Open a terminal on the JetBot HOST")
    print("(SSH or desktop) and run:\n")
    print("    sudo docker restart jetbot_jupyter\n")
    print("Then reload Jupyter Lab and re-run this notebook.")
else:
    print("Camera check: FAILED. Last output:")
    print("\n".join(out.strip().splitlines()[-15:]))
    print()
    print("If this still fails, check (on the HOST):")
    print("  - sudo systemctl status nvargus-daemon")
    print("  - dmesg | grep -iE 'imx|csi|vi '   (loose/reversed ribbon cable)")
    print("  - sudo docker ps                    (a second container also holding the camera)")


## Done

In Jupyter Lab, confirm **Running Terminals and Kernels** only lists this notebook, then open your camera notebook, restart its kernel, and run it top to bottom.